# MTP Project — Download Candidate Models to Google Drive

**Purpose:** Download all four candidate models from HuggingFace Hub to Google Drive for offline use during training sessions.

**Models to download:**
| # | Model | Size | Priority |
|---|-------|------|----------|
| 1 | Qwen/Qwen2.5-3B-Instruct | ~6 GB | ⭐ Recommended first choice |
| 2 | meta-llama/Llama-3.2-3B-Instruct | ~6 GB | Second choice |
| 3 | google/gemma-2-2b-it | ~5 GB | Smallest, fastest |
| 4 | microsoft/Phi-3.5-mini-instruct | ~7.6 GB | Borderline — 3.8B |
| 5 | cross-encoder/nli-deberta-v3-large | ~0.9 GB | ⚙️ NLI scorer (required for Arm C + eval) |

**Total disk needed: ~26 GB** (you have 118 GB available)

**Estimated download time: 30–60 min depending on connection speed**

---
⚠️ **Run cells in order. Do not skip any cell.**

## Cell 0 — Connect To Github

In [2]:
# ── Setup: Mount Drive + Clone Repo ─────────────────────────────────────────
from google.colab import drive, userdata
import os

# Mount Drive
drive.mount('/content/drive')

# Clone repo (uses GitHub token from Colab Secrets)
GITHUB_USER  = "netajik"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO         = "mtp-faithfulness-robustness"

os.chdir('/content')
if not os.path.exists(f'/content/{REPO}'):
    os.system(
        f'git clone https://{GITHUB_USER}:{GITHUB_TOKEN}'
        f'@github.com/{GITHUB_USER}/{REPO}.git'
    )
os.chdir(f'/content/{REPO}')

# Git identity
os.system('git config user.email "kancharapunetaji@email.com"')
os.system('git config user.name "Kancharapu Netaji"')
print(f"✅ Repo ready at /content/{REPO}")

def git_push(message):
    """Save and push all changes to GitHub."""
    os.chdir(f'/content/{REPO}')
    os.system('git add -A')
    result = os.system(f'git commit -m "{message}"')
    if result == 0:
        os.system('git push')
        print(f"✅ Pushed: {message}")
    else:
        print("ℹ️  Nothing new to commit")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Repo ready at /content/mtp-faithfulness-robustness


## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

Mounted at /content/drive
✅ Google Drive mounted


## Cell 2 — Create Project Directory Structure

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/mtp_project"
MODELS_DIR = f"{BASE_DIR}/models"
DATA_DIR = f"{BASE_DIR}/data"
CHECKPOINTS_DIR = f"{BASE_DIR}/checkpoints"
EVAL_DIR = f"{BASE_DIR}/eval"
CACHE_DIR = f"{BASE_DIR}/cache"
LOGS_DIR = f"{BASE_DIR}/logs"

dirs = [BASE_DIR, MODELS_DIR, DATA_DIR, CHECKPOINTS_DIR,
        EVAL_DIR, CACHE_DIR, LOGS_DIR]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  📁 {d}")

print("\n✅ Directory structure created")

  📁 /content/drive/MyDrive/mtp_project
  📁 /content/drive/MyDrive/mtp_project/models
  📁 /content/drive/MyDrive/mtp_project/data
  📁 /content/drive/MyDrive/mtp_project/checkpoints
  📁 /content/drive/MyDrive/mtp_project/eval
  📁 /content/drive/MyDrive/mtp_project/cache
  📁 /content/drive/MyDrive/mtp_project/logs

✅ Directory structure created


## Cell 3 — Install Dependencies

In [ ]:
!pip install -q transformers huggingface_hub accelerate
print("✅ Dependencies installed")

✅ Dependencies installed


## Cell 4 — Get Access



**Make sure you have:**
Requested access to gated models:
   - https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct → Request access
   - https://huggingface.co/google/gemma-2-2b-it → Acknowledge license


In [ ]:
import os
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN_NEW')

if not HF_TOKEN or not HF_TOKEN.startswith('hf_'):
    print("❌ HF_TOKEN not set or invalid")
    print("   Run the one-time setup cell below to create your .env file")
else:
    login(token=HF_TOKEN)
    masked = HF_TOKEN[:8] + '*' * (len(HF_TOKEN) - 8)
    print(f"✅ Logged in to HuggingFace — token: {masked}")


✅ Logged in to HuggingFace — token: hf_CVOXJ*****************************


## Cell 5 — Download Helper Function

In [ ]:
from huggingface_hub import snapshot_download
import time
import os

def download_model(repo_id, save_name, token=None):
    """
    Download a model from HuggingFace Hub to Google Drive.
    Skips if already downloaded.
    """
    save_path = f"{MODELS_DIR}/{save_name}"

    # Check if already downloaded
    if os.path.exists(save_path) and len(os.listdir(save_path)) > 0:
        print(f"⏭️  {save_name} already exists at {save_path} — skipping")
        return save_path

    print(f"\n⬇️  Downloading {repo_id}")
    print(f"   → Saving to: {save_path}")
    start = time.time()

    snapshot_download(
        repo_id=repo_id,
        local_dir=save_path,
        token=token,
        ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*",
                         "rust_model*", "*.ot"],  # skip non-PyTorch weights
    )

    elapsed = time.time() - start
    size_gb = sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files in os.walk(save_path)
        for f in files
    ) / 1e9

    print(f"✅ {save_name} downloaded — {size_gb:.1f} GB in {elapsed/60:.1f} min")
    return save_path

print("✅ Download function ready")

✅ Download function ready


## Cell 6 — Download Model 1: Qwen2.5-3B-Instruct ⭐
**Recommended first choice. No gating — downloads without token.**

In [ ]:
download_model(
    repo_id="Qwen/Qwen2.5-3B-Instruct",
    save_name="Qwen2.5-3B-Instruct",
    token=HF_TOKEN if not HF_TOKEN.startswith("hf_xxx") else None
)


⬇️  Downloading Qwen/Qwen2.5-3B-Instruct
   → Saving to: /content/drive/MyDrive/mtp_project/models/Qwen2.5-3B-Instruct


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Qwen2.5-3B-Instruct downloaded — 6.2 GB in 0.8 min


'/content/drive/MyDrive/mtp_project/models/Qwen2.5-3B-Instruct'

## Cell 7 — Download Model 2: Llama-3.2-3B-Instruct
**Requires HF access request approved at:** https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

In [ ]:
download_model(
    repo_id="meta-llama/Llama-3.2-3B-Instruct",
    save_name="Llama-3.2-3B-Instruct",
    token=HF_TOKEN
)


⬇️  Downloading meta-llama/Llama-3.2-3B-Instruct
   → Saving to: /content/drive/MyDrive/mtp_project/models/Llama-3.2-3B-Instruct


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Llama-3.2-3B-Instruct downloaded — 12.9 GB in 2.5 min


'/content/drive/MyDrive/mtp_project/models/Llama-3.2-3B-Instruct'

## Cell 8 — Download Model 3: Gemma-2-2b-it
**Requires license acknowledgment at:** https://huggingface.co/google/gemma-2-2b-it

In [ ]:
download_model(
    repo_id="google/gemma-2-2b-it",
    save_name="gemma-2-2b-it",
    token=HF_TOKEN
)


⬇️  Downloading google/gemma-2-2b-it
   → Saving to: /content/drive/MyDrive/mtp_project/models/gemma-2-2b-it


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

✅ gemma-2-2b-it downloaded — 5.3 GB in 0.8 min


'/content/drive/MyDrive/mtp_project/models/gemma-2-2b-it'

## Cell 9 — Download Model 4: Phi-3.5-mini-instruct
**3.8B — borderline on T4. Download now, test VRAM fit later.**

In [ ]:
download_model(
    repo_id="microsoft/Phi-3.5-mini-instruct",
    save_name="Phi-3.5-mini-instruct",
    token=HF_TOKEN if not HF_TOKEN.startswith("hf_xxx") else None
)


⬇️  Downloading microsoft/Phi-3.5-mini-instruct
   → Saving to: /content/drive/MyDrive/mtp_project/models/Phi-3.5-mini-instruct


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

✅ Phi-3.5-mini-instruct downloaded — 7.6 GB in 1.1 min


'/content/drive/MyDrive/mtp_project/models/Phi-3.5-mini-instruct'

## Cell 9b — Download DeBERTa-v3-large-mnli (NLI Scorer)
**Required for Arm C offline scoring and faithfulness evaluation.**  
Used to compute `answer_consistency = P_entail(rationale → answer)` in the contrastive loss.  
No gating — downloads without token. Size: ~900 MB.

In [ ]:
# DeBERTa NLI model — used in:
#   1. Arm C offline scoring: score(r,x) = avg(paraphrase_stability, answer_consistency)
#   2. Faithfulness eval: rationale-answer NLI consistency metric
# HuggingFace: https://huggingface.co/cross-encoder/nli-deberta-v3-large

download_model(
    repo_id="cross-encoder/nli-deberta-v3-large",
    save_name="nli-deberta-v3-large",
    token=HF_TOKEN if not HF_TOKEN.startswith("hf_xxx") else None
)


⬇️  Downloading cross-encoder/nli-deberta-v3-large
   → Saving to: /content/drive/MyDrive/mtp_project/models/nli-deberta-v3-large


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

✅ nli-deberta-v3-large downloaded — 9.6 GB in 1.5 min


'/content/drive/MyDrive/mtp_project/models/nli-deberta-v3-large'

## Cell 10 — Verify All Downloads (5 models + NLI scorer)

In [ ]:
import os

models = [
    "Qwen2.5-3B-Instruct",
    "Llama-3.2-3B-Instruct",
    "gemma-2-2b-it",
    "Phi-3.5-mini-instruct",
    "nli-deberta-v3-large",
]

print("=" * 55)
print("MODEL DOWNLOAD STATUS")
print("=" * 55)

total_gb = 0
for model in models:
    path = f"{MODELS_DIR}/{model}"
    if os.path.exists(path):
        # Count files and size
        files = []
        for root, _, fs in os.walk(path):
            files.extend(fs)
        size = sum(
            os.path.getsize(os.path.join(root, f))
            for root, _, fs in os.walk(path)
            for f in fs
        ) / 1e9
        total_gb += size

        # Check key files exist
        has_config = os.path.exists(f"{path}/config.json")
        has_weights = any(
            f.endswith(".safetensors") or f.endswith(".bin")
            for f in files
        )
        has_tokenizer = os.path.exists(f"{path}/tokenizer.json") or \
                        os.path.exists(f"{path}/tokenizer_config.json")

        status = "✅" if (has_config and has_weights and has_tokenizer) else "⚠️ "
        print(f"{status} {model}")
        print(f"     Size: {size:.1f} GB | Files: {len(files)}")
        print(f"     config.json: {'✓' if has_config else '✗'}  "
              f"weights: {'✓' if has_weights else '✗'}  "
              f"tokenizer: {'✓' if has_tokenizer else '✗'}")
    else:
        print(f"❌ {model} — NOT FOUND at {path}")
    print()

print("-" * 55)
print(f"Total downloaded: {total_gb:.1f} GB")

# Check Drive space remaining
import shutil
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive space free:  {free/1e9:.1f} GB")
print("=" * 55)

MODEL DOWNLOAD STATUS
✅ Qwen2.5-3B-Instruct
     Size: 6.2 GB | Files: 26
     config.json: ✓  weights: ✓  tokenizer: ✓

✅ Llama-3.2-3B-Instruct
     Size: 12.9 GB | Files: 34
     config.json: ✓  weights: ✓  tokenizer: ✓

✅ gemma-2-2b-it
     Size: 5.3 GB | Files: 24
     config.json: ✓  weights: ✓  tokenizer: ✓

✅ Phi-3.5-mini-instruct
     Size: 7.6 GB | Files: 42
     config.json: ✓  weights: ✓  tokenizer: ✓

✅ nli-deberta-v3-large
     Size: 9.6 GB | Files: 36
     config.json: ✓  weights: ✓  tokenizer: ✓

-------------------------------------------------------
Total downloaded: 41.6 GB
Drive space free:  57.7 GB


## Cell 11 — Quick Sanity Check: Load Config of Each Model
**Does NOT load weights — just checks configs can be read. Fast.**

In [ ]:
from transformers import AutoConfig

print("=" * 55)
print("CONFIG SANITY CHECK")
print("=" * 55)

for model in models:
    path = f"{MODELS_DIR}/{model}"
    if not os.path.exists(path):
        print(f"❌ {model} — path not found, skip")
        continue
    try:
        cfg = AutoConfig.from_pretrained(path, trust_remote_code=True)
        params_b = getattr(cfg, 'num_parameters', None)
        hidden  = getattr(cfg, 'hidden_size', '?')
        layers  = getattr(cfg, 'num_hidden_layers', '?')
        vocab   = getattr(cfg, 'vocab_size', '?')
        print(f"✅ {model}")
        print(f"     hidden_size={hidden}  layers={layers}  vocab={vocab}")
    except Exception as e:
        print(f"⚠️  {model} — config error: {e}")
    print()

print("All configs checked.")

CONFIG SANITY CHECK
✅ Qwen2.5-3B-Instruct
     hidden_size=2048  layers=36  vocab=151936

✅ Llama-3.2-3B-Instruct
     hidden_size=3072  layers=28  vocab=128256

✅ gemma-2-2b-it
     hidden_size=2304  layers=26  vocab=256000



This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


✅ Phi-3.5-mini-instruct
     hidden_size=3072  layers=32  vocab=32064

✅ nli-deberta-v3-large
     hidden_size=1024  layers=24  vocab=128100

All configs checked.


## Cell 12 — Save Download Log to Drive

In [ ]:
import json
from datetime import datetime

log = {
    "timestamp": datetime.now().isoformat(),
    "models_dir": MODELS_DIR,
    "models": {}
}

for model in models:
    path = f"{MODELS_DIR}/{model}"
    if os.path.exists(path):
        size = sum(
            os.path.getsize(os.path.join(root, f))
            for root, _, fs in os.walk(path)
            for f in fs
        ) / 1e9
        log["models"][model] = {
            "path": path,
            "size_gb": round(size, 2),
            "status": "downloaded"
        }
    else:
        log["models"][model] = {"status": "not_downloaded"}

log_path = f"{BASE_DIR}/download_log.json"
with open(log_path, "w") as f:
    json.dump(log, f, indent=2)

print(f"✅ Download log saved to: {log_path}")
print(json.dumps(log, indent=2))

✅ Download log saved to: /content/drive/MyDrive/mtp_project/download_log.json
{
  "timestamp": "2026-04-17T16:48:27.109208",
  "models_dir": "/content/drive/MyDrive/mtp_project/models",
  "models": {
    "Qwen2.5-3B-Instruct": {
      "path": "/content/drive/MyDrive/mtp_project/models/Qwen2.5-3B-Instruct",
      "size_gb": 6.18,
      "status": "downloaded"
    },
    "Llama-3.2-3B-Instruct": {
      "path": "/content/drive/MyDrive/mtp_project/models/Llama-3.2-3B-Instruct",
      "size_gb": 12.86,
      "status": "downloaded"
    },
    "gemma-2-2b-it": {
      "path": "/content/drive/MyDrive/mtp_project/models/gemma-2-2b-it",
      "size_gb": 5.25,
      "status": "downloaded"
    },
    "Phi-3.5-mini-instruct": {
      "path": "/content/drive/MyDrive/mtp_project/models/Phi-3.5-mini-instruct",
      "size_gb": 7.64,
      "status": "downloaded"
    },
    "nli-deberta-v3-large": {
      "path": "/content/drive/MyDrive/mtp_project/models/nli-deberta-v3-large",
      "size_gb": 9.65,
  

---
## ✅ All Done!

Your models are saved at:
```
MyDrive/
└── mtp_project/
    ├── models/
    │   ├── Qwen2.5-3B-Instruct/
    │   ├── Llama-3.2-3B-Instruct/
    │   ├── gemma-2-2b-it/
    │   └── Phi-3.5-mini-instruct/
    ├── checkpoints/    ← training checkpoints will go here
    ├── data/           ← Alpaca dataset will go here
    ├── eval/           ← evaluation scripts and results
    ├── cache/          ← rationale scores cache
    └── logs/           ← training logs
```

**Next step:** Run the model selection pilot notebook to identify which model to use for training.

**Before closing this session:** Make sure Cell 10 shows ✅ for all four models.

In [3]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# List all available models
models = client.models.list()
model_ids = sorted([m.id for m in models.data])

# Filter relevant ones
keywords = ['gpt-4', 'gpt-5', 'o1', 'o3', 'o4']
relevant = [m for m in model_ids if any(k in m for k in keywords)]

print("=" * 50)
print("AVAILABLE MODELS TODAY")
print("=" * 50)
for m in relevant:
    print(f"  {m}")
print(f"\nTotal relevant: {len(relevant)}")

AVAILABLE MODELS TODAY
  gpt-4.1
  gpt-4.1-2025-04-14
  gpt-4.1-mini
  gpt-4.1-mini-2025-04-14
  gpt-4.1-nano
  gpt-4.1-nano-2025-04-14
  gpt-4o
  gpt-4o-2024-05-13
  gpt-4o-2024-08-06
  gpt-4o-2024-11-20
  gpt-4o-audio-preview
  gpt-4o-audio-preview-2024-12-17
  gpt-4o-audio-preview-2025-06-03
  gpt-4o-mini
  gpt-4o-mini-2024-07-18
  gpt-4o-mini-audio-preview
  gpt-4o-mini-audio-preview-2024-12-17
  gpt-4o-mini-search-preview
  gpt-4o-mini-search-preview-2025-03-11
  gpt-4o-mini-transcribe
  gpt-4o-mini-transcribe-2025-03-20
  gpt-4o-mini-transcribe-2025-12-15
  gpt-4o-mini-tts
  gpt-4o-mini-tts-2025-03-20
  gpt-4o-mini-tts-2025-12-15
  gpt-4o-search-preview
  gpt-4o-search-preview-2025-03-11
  gpt-4o-transcribe
  gpt-4o-transcribe-diarize
  gpt-5
  gpt-5-2025-08-07
  gpt-5-chat-latest
  gpt-5-codex
  gpt-5-mini
  gpt-5-mini-2025-08-07
  gpt-5-nano
  gpt-5-nano-2025-08-07
  gpt-5-pro
  gpt-5-pro-2025-10-06
  gpt-5-search-api
  gpt-5-search-api-2025-10-14
  gpt-5.1
  gpt-5.1-2025-11-13